# Softmax regression from scratch

Goal: implement multinomial logistic regression in NumPy (loss, gradient, gradient descent), check the gradient numerically, and validate the result against scikit-learn. The code is in `src/softmax.py`.

In [1]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from src.data import load_splits
from src.metrics import evaluate
from src.softmax import SoftmaxRegression, loss_and_grad, softmax

sns.set_theme(style='whitegrid')

data, y_train, y_test = load_splits()

## Gradient check

Compare the analytic gradient with a central finite difference $\frac{L(\theta+\varepsilon e_j) - L(\theta-\varepsilon e_j)}{2\varepsilon}$ on a small random problem. The relative error should be around $10^{-8}$ or less.

In [2]:
rng = np.random.default_rng(0)
n, p, K = 50, 4, 5
X = rng.normal(size=(n, p))
y_idx = rng.integers(0, K, n)
sw = rng.uniform(0.5, 2.0, n)
W0, b0, l2 = rng.normal(size=(p, K)), rng.normal(size=K), 0.1

def f(theta):
    W, b = theta[:p * K].reshape(p, K), theta[p * K:]
    return loss_and_grad(W, b, X, y_idx, sw, l2)[0]

theta = np.concatenate([W0.ravel(), b0])
_, dW, db = loss_and_grad(W0, b0, X, y_idx, sw, l2)
analytic = np.concatenate([dW.ravel(), db])

eps = 1e-6
numeric = np.array([(f(theta + eps * e) - f(theta - eps * e)) / (2 * eps) for e in np.eye(len(theta))])

rel_err = np.linalg.norm(analytic - numeric) / (np.linalg.norm(analytic) + np.linalg.norm(numeric))
print('relative error :', rel_err)
assert rel_err < 1e-6

relative error : 8.097608723760522e-10


## Training

On the scaled training set, with and without class weights, for both feature sets. The loss must decrease.

In [3]:
results, models = {}, {}
for fs, d in data.items():
    for cw in [None, 'balanced']:
        model = SoftmaxRegression(learning_rate=0.5, n_iter=5000, l2=1e-3, class_weight=cw)
        model.fit(d['X_train_s'], y_train)
        pred = model.predict(d['X_test_s'])
        summary, table = evaluate(y_test, pred)
        models[(fs, cw)] = model
        results[(fs, cw or 'none')] = {**summary, 'recall_1': table.loc[1, 'recall'], 'recall_2': table.loc[2, 'recall']}

pd.DataFrame(results).T.round(3)

KeyboardInterrupt: 

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for (fs, cw), m in models.items():
    ax.plot(m.loss_history_, label=f'{fs}, {cw or "no weights"}')
ax.set_xlabel('iteration'); ax.set_ylabel('loss'); ax.set_yscale('log'); ax.legend()
plt.show()

assert all(m.loss_history_[-1] < m.loss_history_[0] for m in models.values())

## Validation against scikit-learn

scikit-learn minimizes $C\sum_i w_i\ell_i + \tfrac12\lVert W\rVert^2$, which is our loss multiplied by $nC$ when $\lambda = 1/(nC)$. With `C = 1 / (l2 * n)` and the same class weights, both solve the same problem, so the coefficients and the predictions should match closely

In [ ]:
rows = []
d = data['salary']
n_train = len(y_train)
for cw in [None, 'balanced']:
    ours = models[('salary', cw)]
    sk = LogisticRegression(C=1 / (1e-3 * n_train), class_weight=cw, max_iter=5000).fit(d['X_train_s'], y_train)
    p_ours, p_sk = ours.predict(d['X_test_s']), sk.predict(d['X_test_s'])
    rows.append({
        'class_weight': cw or 'none',
        'max |W diff|': np.abs(ours.W_ - sk.coef_.T).max(),
        'max |b diff|': np.abs(ours.b_ - sk.intercept_).max(),
        'same predictions': (p_ours == p_sk).mean(),
        'macro_f1 ours': evaluate(y_test, p_ours)[0]['macro_f1'],
        'macro_f1 sklearn': evaluate(y_test, p_sk)[0]['macro_f1'],
    })
pd.DataFrame(rows).set_index('class_weight').round(4)